<a href="https://colab.research.google.com/github/catrina-llamas-1/Cats-Repository/blob/main/brochure_pdf_excel_extractor_no_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brochure PDF → Excel Extractor (no API)

This is a **fully local, no-API** version of the brochure extractor: it reads a commercial real estate brochure PDF and extracts each listing into an Excel sheet with these columns:

`Property Name | Property Address | Asking Rent | Additional Rent | Headlease/Sublease | Parking Information | Suite/Floor | Area (SF) | Comments | Submarket | Contact`

**How it works:** instead of sending pages to Claude, this notebook reads each word's exact (x, y) position on the page with `pdfplumber` and reconstructs the document's columns from those coordinates — a left-hand label column (Asking rate/Sale Price, Additional rent, Parking) and a right-hand table (Suite/floor, Area (sf), Comments). It was built and tuned against Avison Young's "Office Exclusive Listings" brochure template specifically (validated against a real June 2026 issue: 215 listing rows across 59 properties, extracted correctly).

**Trade-off vs. the API version:** no cost, no API key, fully offline, deterministic. In exchange, it's tuned to *this* brochure's layout — column x-positions, page header/footer text, and label wording ("Asking rate (psf)", "Sale Price", "Additional rent (psf)", "Parking", "Suite/floor", "Area (sf)", "Comments", "Contact:") are all hardcoded assumptions. If a future issue of this brochure changes its template, or you point this at a brochure from a different broker, the constants in the Configuration cell below (and possibly the parsing logic itself) will need retuning.

### Known limitations (accepted trade-offs for a deterministic, no-cost parser)

- **Comments are attached per-suite only when the source text explicitly prefixes them** with the suite/floor code (e.g. `"240: Raw space, ready for tenant improvements."`). When a property's comments are one general paragraph with no such prefixes, the *entire* comments blob is repeated on every suite/floor row for that property (this matches how the source document actually presents that information).
- A suite/floor label that wraps across two lines (e.g. `"Standalone"` / `"building"`), or a sub-building divider label in a multi-building listing (e.g. `"Revillon"` / `"Boardwalk"`), each become their own thin row with an empty area — a minor artifact rather than being merged (incorrectly) into a neighboring row.
- A rare "compound" area cell that breaks one suite down into multiple use types (e.g. `Office: 27,963 / Warehouse: 25,631 / Lab: 25,281`) is not fully captured — only the first line is read. Worth a manual spot-check if a brochure includes mixed-use sale listings like this.
- Contact names must fit on the same visual line as the property name/address; a contact list that wraps onto its own line is not currently handled.

**Requirements:** `pip install pdfplumber openpyxl pandas` (done in the first code cell). No API key needed.


## 0. Configuration — edit this section

In [ ]:
# ── File paths ──────────────────────────────────────────────────────────────
PDF_PATH    = "brochure.pdf"                    # Path to the source brochure PDF
OUTPUT_PATH = "brochure_listings_no_api.xlsx"   # Where to write the extracted Excel sheet

# ── Layout constants tuned to this brochure's template ───────────────────────
# These are starting points: for each property block, the parser first tries
# to find the real "Asking rate (psf)"/"Sale Price" + "Suite/floor" +
# "Area (sf)" + "Comments" header row on the page and reads the true x
# positions from it. These defaults are only a fallback for the rare case
# that header row can't be found.
RATE_LABEL_X_DEFAULT = 160.8    # left column: Asking rate / Sale Price / Additional rent / Parking
SUITE_X_DEFAULT = 289.7         # Suite/floor column
AREA_X_DEFAULT = 343.0          # Area (sf) column
COMMENTS_X_DEFAULT = 390.0      # Comments column

LEFT_MARGIN_MAX = 100.0    # x0 below this = submarket header / property name line
TOP_CHROME_MAX = 75.0      # page title + submarket nav bar (skipped on every page)
BOTTOM_CHROME_MIN = 750.0  # footer page number / legend row (skipped on every page)

## 1. Upload the brochure (Google Colab only)

If you're running this in Google Colab, run the cell below to upload the brochure PDF from your computer — it overrides `PDF_PATH` from the config cell above with the uploaded file. If you're running locally (Jupyter, VS Code, etc.), skip this cell and just point `PDF_PATH` at a file already on disk.

In [ ]:
try:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        PDF_PATH = next(iter(uploaded))
        print(f"Uploaded {PDF_PATH!r} - PDF_PATH updated.")
except ImportError:
    print("Not running in Google Colab - using the PDF_PATH set above.")

## 2. Install and import dependencies

In [ ]:
%pip install -q pdfplumber openpyxl pandas

In [ ]:
import re

import pandas as pd
import pdfplumber
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

## 3. Load words and reconstruct visual lines

`pdfplumber` gives us every word's exact position on the page. We drop page chrome (title bar, submarket nav, footer/legend) by vertical position, then offset each page's coordinates so the whole document reads as one continuous, page-break-safe stream — a property's table is never accidentally split by a page boundary.

In [ ]:
PAGE_ROW_OFFSET = 2000.0  # larger than any single page's height, keeps ordering monotonic


def load_words(pdf_path):
    """Return a flat list of word dicts across the whole document, with a
    global monotonically increasing 'top' so page boundaries don't break
    vertical ordering, and page chrome (title/nav/footer/legend) removed."""
    all_words = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages):
            for w in page.extract_words():
                if w["top"] < TOP_CHROME_MAX or w["top"] > BOTTOM_CHROME_MIN:
                    continue
                all_words.append(
                    {
                        "text": w["text"],
                        "x0": w["x0"],
                        "top": w["top"] + page_index * PAGE_ROW_OFFSET,
                        "page": page_index + 1,
                    }
                )
    return all_words


def group_into_lines(words, tol=3.0):
    """Group words sharing (approximately) the same 'top' into visual lines,
    each returned as a list of words sorted left-to-right."""
    words = sorted(words, key=lambda w: (w["top"], w["x0"]))
    lines = []
    current = []
    current_top = None
    for w in words:
        if current_top is None or abs(w["top"] - current_top) <= tol:
            current.append(w)
            current_top = w["top"] if current_top is None else current_top
        else:
            lines.append(current)
            current = [w]
            current_top = w["top"]
    if current:
        lines.append(current)
    return lines


def line_text(line_words):
    return " ".join(w["text"] for w in line_words)

## 4. Identify submarket headers and property lines

In [ ]:
def is_all_caps_line(line_words):
    letters = "".join(ch for w in line_words for ch in w["text"] if ch.isalpha())
    return bool(letters) and letters.isupper()


def is_property_line(line_words):
    """A property name/address line: left-margin indent, mixed case (not a
    submarket header), and not a stray leftover chrome line."""
    if not line_words:
        return False
    if line_words[0]["x0"] >= LEFT_MARGIN_MAX:
        return False
    if is_all_caps_line(line_words):
        return False
    if line_text(line_words).startswith("Contact:"):
        return False
    return True


def find_contact_boundary(line_words):
    """Locate the literal 'Contact:' token on a line, if present, and return
    its x0. Using the token itself (rather than a fixed x-threshold) is
    robust to how far right it sits, which varies with address/name length."""
    for w in line_words:
        if w["text"] == "Contact:":
            return w["x0"]
    return None


def split_property_line(line_words):
    """Split a property line into (name, address, is_sublease), using only
    the words left of 'Contact:' (if present on the same line)."""
    boundary = find_contact_boundary(line_words)
    left_words = [w for w in line_words if boundary is None or w["x0"] < boundary]
    text = line_text(left_words).strip()
    # Drop empty segments (e.g. a trailing "|" with nothing after it means
    # this listing has no separate name - just a bare street address).
    parts = [p.strip() for p in text.split("|")]
    parts = [p for p in parts if p]
    is_sublease = False
    if parts and parts[-1].upper() == "SUBLEASE":
        is_sublease = True
        parts = parts[:-1]
    if len(parts) >= 2:
        name, address = parts[0], parts[1]
    elif len(parts) == 1:
        name, address = "", parts[0]
    else:
        name, address = "", ""
    return name, address, is_sublease


def extract_contact(line_words):
    boundary = find_contact_boundary(line_words)
    if boundary is None:
        return ""
    contact_words = [w for w in line_words if w["x0"] >= boundary]
    text = line_text(contact_words)
    text = re.sub(r"^Contact:\s*", "", text).strip()
    return text

## 5. Parse a property's body: label column + suite/floor table

Each property block has a left-hand label column (Asking rate/Sale Price, Additional rent, Parking) and a right-hand table (Suite/floor, Area (sf), Comments). The column x-positions are read from that property's own header row when possible, so small page-to-page drift doesn't break the split.

In [ ]:
def find_column_anchors(body_lines):
    """Look for the 'Asking rate (psf)'/'Sale Price' + 'Suite/floor' +
    'Area (sf)' + 'Comments' header row within a property's body lines.
    Falls back to the document-wide default x-positions if not found.
    Returns the header line's 'top' too, so callers can exclude it from
    the data rows."""
    rate_x = suite_x = area_x = comments_x = None
    is_sale = False
    header_top = None
    for line_words in body_lines:
        text = line_text(line_words)
        if "Suite/floor" in text and ("Asking rate" in text or "Sale Price" in text):
            for w in line_words:
                if w["text"] == "Suite/floor":
                    suite_x = w["x0"]
                elif w["text"] == "Area":
                    area_x = w["x0"]
                elif w["text"] == "Comments":
                    comments_x = w["x0"]
                elif w["text"] in ("Asking", "Sale"):
                    rate_x = w["x0"]
            is_sale = "Sale Price" in text
            header_top = line_words[0]["top"]
            break
    return (
        rate_x or RATE_LABEL_X_DEFAULT,
        suite_x or SUITE_X_DEFAULT,
        area_x or AREA_X_DEFAULT,
        comments_x or COMMENTS_X_DEFAULT,
        is_sale,
        header_top,
    )


def parse_label_column(body_lines, rate_x, suite_x):
    """Reconstruct the Asking rate/Sale Price, Additional rent, and Parking
    text blocks from the left-hand label column."""
    label_lines = []
    for line_words in body_lines:
        cols = [w for w in line_words if rate_x - 5 <= w["x0"] < suite_x - 5]
        if cols:
            label_lines.append(line_text(cols))

    full_text = "\n".join(label_lines)
    heading_pattern = r"(Additional rent \(psf\)|Asking rate \(psf\)|Sale Price|Parking)"
    parts = re.split(heading_pattern, full_text)
    fields = {}
    for i in range(1, len(parts), 2):
        heading = parts[i].strip()
        value = parts[i + 1].strip(" \n") if i + 1 < len(parts) else ""
        value = " ".join(line.strip() for line in value.splitlines() if line.strip())
        fields.setdefault(heading, value)
    return fields


def parse_table(body_lines, suite_x, area_x, comments_x):
    """Return a list of {'suite_floor', 'area_sf'} rows (in document order)
    plus the raw comments lines from the right-hand column."""
    rows = []
    comment_lines = []
    for line_words in body_lines:
        suite_words = [w for w in line_words if suite_x - 5 <= w["x0"] < area_x - 5]
        area_words = [w for w in line_words if area_x - 5 <= w["x0"] < comments_x - 5]
        comments_words = [w for w in line_words if w["x0"] >= comments_x - 5]

        suite_text = line_text(suite_words).strip()
        area_text = line_text(area_words).strip()

        if suite_text:
            # Any line with suite/floor-column text starts a new row. When a
            # suite name wraps across two lines (e.g. "Standalone" /
            # "building") or a sub-group divider label appears with no area
            # of its own (e.g. "Boardwalk"), it becomes its own thin row
            # rather than being merged into a neighboring row - that avoids
            # ever corrupting another row's suite/floor value.
            rows.append({"suite_floor": suite_text, "area_sf": area_text})

        if comments_words:
            comment_lines.append(line_text(comments_words))

    return rows, comment_lines


def assign_comments(rows, comment_lines):
    """Best-effort per-suite comment attribution: if comment lines are
    explicitly prefixed with a suite/floor code ('240: Raw space...'),
    split accordingly. Otherwise every row gets the full comments blob."""
    suite_codes = {r["suite_floor"] for r in rows if r["suite_floor"]}
    escaped = sorted((re.escape(c) for c in suite_codes if c), key=len, reverse=True)
    per_suite = {}
    if escaped:
        marker_re = re.compile(r"^(" + "|".join(escaped) + r")\s*:\s*(.*)$")
        current_code = None
        for line in comment_lines:
            m = marker_re.match(line)
            if m:
                current_code = m.group(1)
                per_suite.setdefault(current_code, []).append(m.group(2))
            elif current_code is not None:
                per_suite[current_code].append(line)
            # else: pre-marker preamble text is dropped (rare)

    if per_suite:
        for r in rows:
            r["comments"] = " ".join(per_suite.get(r["suite_floor"], [])).strip()
    else:
        blob = " ".join(comment_lines).strip()
        for r in rows:
            r["comments"] = blob
    return rows

## 6. Walk the document and assemble listings

In [ ]:
def parse_document(pdf_path):
    words = load_words(pdf_path)
    lines = group_into_lines(words)

    listings = []
    current_submarket = ""
    current_property = None  # dict with name/address/sublease/contact/body_lines

    def flush():
        nonlocal current_property
        if current_property is None:
            return
        rate_x, suite_x, area_x, comments_x, is_sale, header_top = find_column_anchors(
            current_property["body_lines"]
        )
        # The label column (Asking rate/Sale Price heading + value, Additional
        # rent, Parking) needs the header row too, since that's where the
        # "Asking rate (psf)" / "Sale Price" heading text itself lives.
        label_fields = parse_label_column(current_property["body_lines"], rate_x, suite_x)
        # But the table (suite/floor + area + comments) must NOT include the
        # header row itself, or "Suite/floor" / "Area (sf)" / "Comments"
        # get mistaken for a data row.
        table_lines = [
            ln
            for ln in current_property["body_lines"]
            if header_top is None or abs(ln[0]["top"] - header_top) > 0.5
        ]
        rows, comment_lines = parse_table(table_lines, suite_x, area_x, comments_x)
        rows = assign_comments(rows, comment_lines)

        asking_rent = label_fields.get("Sale Price") or label_fields.get("Asking rate (psf)", "")
        additional_rent = label_fields.get("Additional rent (psf)", "")
        parking = label_fields.get("Parking", "")

        if is_sale:
            headlease_sublease = "Sale"
        elif current_property["is_sublease"]:
            headlease_sublease = "Sublease"
        else:
            headlease_sublease = "Headlease"

        if not rows:
            rows = [{"suite_floor": "", "area_sf": "", "comments": " ".join(comment_lines)}]

        has_signal = bool(
            asking_rent or additional_rent or parking
            or any(r["suite_floor"] or r["area_sf"] for r in rows)
        )
        if not has_signal:
            # Nothing recognizable was found in this block (e.g. a cover
            # page or table-of-contents page) - skip it rather than emit
            # junk rows.
            current_property = None
            return

        for row in rows:
            listings.append(
                {
                    "property_name": current_property["name"],
                    "property_address": current_property["address"],
                    "asking_rent": asking_rent,
                    "additional_rent": additional_rent,
                    "headlease_sublease": headlease_sublease,
                    "parking_information": parking,
                    "suite_floor": row["suite_floor"],
                    "area_sf": row["area_sf"],
                    "comments": row["comments"],
                    "submarket": current_property["submarket"],
                    "contact": current_property["contact"],
                }
            )
        current_property = None

    for line_words in lines:
        if not line_words:
            continue
        if line_words[0]["x0"] >= LEFT_MARGIN_MAX:
            # Body content of the current property block
            if current_property is not None:
                current_property["body_lines"].append(line_words)
            continue

        text = line_text(line_words)
        if is_all_caps_line(line_words):
            flush()
            current_submarket = text
            continue

        if is_property_line(line_words):
            flush()
            name, address, is_sublease = split_property_line(line_words)
            contact = extract_contact(line_words)
            current_property = {
                "name": name,
                "address": address,
                "is_sublease": is_sublease,
                "contact": contact,
                "submarket": current_submarket,
                "body_lines": [],
            }
            continue

        # Any other left-margin line before a property block has started is
        # page chrome that slipped through the top/bottom filters - ignore it.

    flush()
    return listings

## 7. Run the parser and build the DataFrame

In [ ]:
COLUMNS = [
    "Property Name",
    "Property Address",
    "Asking Rent",
    "Additional Rent",
    "Headlease/Sublease",
    "Parking Information",
    "Suite/Floor",
    "Area (SF)",
    "Comments",
    "Submarket",
    "Contact",
]

FIELD_TO_COLUMN = {
    "property_name": "Property Name",
    "property_address": "Property Address",
    "asking_rent": "Asking Rent",
    "additional_rent": "Additional Rent",
    "headlease_sublease": "Headlease/Sublease",
    "parking_information": "Parking Information",
    "suite_floor": "Suite/Floor",
    "area_sf": "Area (SF)",
    "comments": "Comments",
    "submarket": "Submarket",
    "contact": "Contact",
}

listings = parse_document(PDF_PATH)
print(f"Extracted {len(listings)} listing row(s) from {PDF_PATH}")

rows = [{FIELD_TO_COLUMN[k]: v for k, v in listing.items()} for listing in listings]
df = pd.DataFrame(rows, columns=COLUMNS)
df

## 8. Write to Excel

In [ ]:
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Listings")

    ws = writer.sheets["Listings"]

    # Bold header row + freeze it
    for cell in ws[1]:
        cell.font = Font(bold=True)
    ws.freeze_panes = "A2"

    # Reasonable column widths, with wrapped text for the long free-form columns
    wrap_columns = {"Parking Information", "Comments"}
    widths = {
        "Property Name": 24,
        "Property Address": 26,
        "Asking Rent": 20,
        "Additional Rent": 16,
        "Headlease/Sublease": 16,
        "Parking Information": 40,
        "Suite/Floor": 12,
        "Area (SF)": 12,
        "Comments": 50,
        "Submarket": 20,
        "Contact": 26,
    }
    for col_index, col_name in enumerate(COLUMNS, start=1):
        letter = get_column_letter(col_index)
        ws.column_dimensions[letter].width = widths.get(col_name, 18)
        if col_name in wrap_columns:
            for cell in ws[letter][1:]:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

print(f"Wrote {len(df)} rows to {OUTPUT_PATH}")